## 1. A model, stripped to its essence

Forget chat windows and personalities for a moment. Underneath everything, a language model is one thing: a function that receives some text and hands back some text. It runs **once**, produces its answer, and holds on to **nothing**. Ask it something a second time and it has no clue it ever met you.

In [60]:
def stateless_reply(prompt: str) -> str:
    return f"(some llm response mimicking the model's output) {prompt}"

In [61]:
print(stateless_reply("Who is the PM of UK"))

(some llm response mimicking the model's output) Who is the PM of UK


In [62]:
print(stateless_reply("who was before him"))

(some llm response mimicking the model's output) who was before him


In [66]:
import os
from dotenv import load_dotenv
from groq import Groq, APIConnectionError, APIStatusError, AuthenticationError, RateLimitError

# Loads .env from the notebook's current working directory (run Jupyter from project root).
load_dotenv()

MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")

def stateless_groq_demo(prompt: str) -> str:
    if not prompt.strip():
        raise ValueError("Prompt must not be empty.")

    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        raise ValueError("GROQ_API_KEY not found. Add it to your .env file or environment.")

    client = Groq(api_key=api_key)
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt.strip()}],
        )
    except AuthenticationError as exc:
        raise RuntimeError("Groq rejected the API key. Check that it is valid and active.") from exc
    except RateLimitError as exc:
        raise RuntimeError("Groq rate limit or quota reached. Check your account limits.") from exc
    except APIConnectionError as exc:
        raise RuntimeError("Could not connect to Groq. Check your internet connection.") from exc
    except APIStatusError as exc:
        raise RuntimeError(f"Groq returned HTTP {exc.status_code}: {exc}") from exc

    return response.choices[0].message.content or "(The model returned an empty response.)"


In [64]:
stateless_groq_demo("Who was the PM of UK before 2020 ? ")

'Before 2020, the United Kingdom had several Prime Ministers. Here’s a concise list of the most recent ones leading up to 2020:\n\n| Term | Prime Minister | Party |\n|------|----------------|-------|\n| **2016\u202f–\u202f2019** | **Theresa May** | Conservative |\n| **2010\u202f–\u202f2016** | **David Cameron** | Conservative |\n| **2007\u202f–\u202f2010** | **Gordon\u202fBrown** | Labour |\n| **2005\u202f–\u202f2007** | **Tony\u202fBlair** (until June\u202f2007) | Labour |\n| **1997\u202f–\u202f2005** | **Tony\u202fBlair** (full term) | Labour |\n| **1992\u202f–\u202f1997** | **John Major** | Conservative |\n| **1979\u202f–\u202f1990** | **Margaret Thatcher** | Conservative |\n| **1974\u202f–\u202f1976** | **Harold Wilson** (second term) | Labour |\n| **1970\u202f–\u202f1974** | **Edward Heath** | Conservative |\n\n**Key points:**\n\n- **Theresa May** was the immediate predecessor to Boris Johnson, who took office in July\u202f2019 and was the PM throughout 2020.\n- The most recent Co

In [67]:
stateless_groq_demo("Who was before him")

'I’m not sure which person you’re referring to. Could you let me know who “him” is, or give a bit more context? Then I’ll be happy to answer your question.'